Install Pyspark

In [9]:
!pip install pyspark

Dataset

In [10]:
import pandas as pd

data = {
    "device_id":[101,101,102,102,103,103,104,104,105,105],
    "device_name":[
        "AC Unit","AC Unit",
        "Server Rack","Server Rack",
        "LED Lights","LED Lights",
        "Projector","Projector",
        "Printer","Printer"
    ],
    "timestamp":[
        "2026-06-01 09:00:00",
        "2026-06-01 20:00:00",
        "2026-06-01 11:00:00",
        "2026-06-01 22:00:00",
        "2026-06-01 08:00:00",
        "2026-06-01 21:00:00",
        "2026-06-01 14:00:00",
        "2026-06-01 19:00:00",
        "2026-06-01 10:00:00",
        "2026-06-01 23:00:00"
    ],
    "energy_kwh":[
        8.5,
        5.2,
        18.0,
        12.5,
        3.0,
        2.0,
        6.5,
        5.8,
        4.5,
        3.5
    ]
}

df = pd.DataFrame(data)

df.to_csv("sensor_logs.csv", index=False)

print(df)

   device_id  device_name            timestamp  energy_kwh
0        101      AC Unit  2026-06-01 09:00:00         8.5
1        101      AC Unit  2026-06-01 20:00:00         5.2
2        102  Server Rack  2026-06-01 11:00:00        18.0
3        102  Server Rack  2026-06-01 22:00:00        12.5
4        103   LED Lights  2026-06-01 08:00:00         3.0
5        103   LED Lights  2026-06-01 21:00:00         2.0
6        104    Projector  2026-06-01 14:00:00         6.5
7        104    Projector  2026-06-01 19:00:00         5.8
8        105      Printer  2026-06-01 10:00:00         4.5
9        105      Printer  2026-06-01 23:00:00         3.5


Start Spark Session

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DeviceAggregation") \
    .getOrCreate()

Load Data(CSV)

In [12]:
df = spark.read.csv(
    "sensor_logs.csv",
    header=True,
    inferSchema=True
)

df.show()

+---------+-----------+-------------------+----------+
|device_id|device_name|          timestamp|energy_kwh|
+---------+-----------+-------------------+----------+
|      101|    AC Unit|2026-06-01 09:00:00|       8.5|
|      101|    AC Unit|2026-06-01 20:00:00|       5.2|
|      102|Server Rack|2026-06-01 11:00:00|      18.0|
|      102|Server Rack|2026-06-01 22:00:00|      12.5|
|      103| LED Lights|2026-06-01 08:00:00|       3.0|
|      103| LED Lights|2026-06-01 21:00:00|       2.0|
|      104|  Projector|2026-06-01 14:00:00|       6.5|
|      104|  Projector|2026-06-01 19:00:00|       5.8|
|      105|    Printer|2026-06-01 10:00:00|       4.5|
|      105|    Printer|2026-06-01 23:00:00|       3.5|
+---------+-----------+-------------------+----------+



Extract Hour from Timestamp

In [13]:
from pyspark.sql.functions import hour

df = df.withColumn(
    "hour",
    hour("timestamp")
)

df.show()

+---------+-----------+-------------------+----------+----+
|device_id|device_name|          timestamp|energy_kwh|hour|
+---------+-----------+-------------------+----------+----+
|      101|    AC Unit|2026-06-01 09:00:00|       8.5|   9|
|      101|    AC Unit|2026-06-01 20:00:00|       5.2|  20|
|      102|Server Rack|2026-06-01 11:00:00|      18.0|  11|
|      102|Server Rack|2026-06-01 22:00:00|      12.5|  22|
|      103| LED Lights|2026-06-01 08:00:00|       3.0|   8|
|      103| LED Lights|2026-06-01 21:00:00|       2.0|  21|
|      104|  Projector|2026-06-01 14:00:00|       6.5|  14|
|      104|  Projector|2026-06-01 19:00:00|       5.8|  19|
|      105|    Printer|2026-06-01 10:00:00|       4.5|  10|
|      105|    Printer|2026-06-01 23:00:00|       3.5|  23|
+---------+-----------+-------------------+----------+----+



Peak vs Off-Peak Classification

In [14]:
from pyspark.sql.functions import when

df = df.withColumn(
    "usage_type",
    when(
        (df.hour >= 9) & (df.hour < 18),
        "Peak"
    ).otherwise("Off-Peak")
)

df.show()

+---------+-----------+-------------------+----------+----+----------+
|device_id|device_name|          timestamp|energy_kwh|hour|usage_type|
+---------+-----------+-------------------+----------+----+----------+
|      101|    AC Unit|2026-06-01 09:00:00|       8.5|   9|      Peak|
|      101|    AC Unit|2026-06-01 20:00:00|       5.2|  20|  Off-Peak|
|      102|Server Rack|2026-06-01 11:00:00|      18.0|  11|      Peak|
|      102|Server Rack|2026-06-01 22:00:00|      12.5|  22|  Off-Peak|
|      103| LED Lights|2026-06-01 08:00:00|       3.0|   8|  Off-Peak|
|      103| LED Lights|2026-06-01 21:00:00|       2.0|  21|  Off-Peak|
|      104|  Projector|2026-06-01 14:00:00|       6.5|  14|      Peak|
|      104|  Projector|2026-06-01 19:00:00|       5.8|  19|  Off-Peak|
|      105|    Printer|2026-06-01 10:00:00|       4.5|  10|      Peak|
|      105|    Printer|2026-06-01 23:00:00|       3.5|  23|  Off-Peak|
+---------+-----------+-------------------+----------+----+----------+



Peak vs Off-Peak Usage Per Device

In [15]:
from pyspark.sql.functions import sum

usage_summary = df.groupBy(
    "device_name",
    "usage_type"
).agg(
    sum("energy_kwh").alias("total_energy")
)

print("Peak vs Off-Peak Usage")

usage_summary.show()

Peak vs Off-Peak Usage
+-----------+----------+------------+
|device_name|usage_type|total_energy|
+-----------+----------+------------+
|    AC Unit|      Peak|         8.5|
|    Printer|  Off-Peak|         3.5|
|  Projector|      Peak|         6.5|
| LED Lights|  Off-Peak|         5.0|
|    AC Unit|  Off-Peak|         5.2|
|  Projector|  Off-Peak|         5.8|
|    Printer|      Peak|         4.5|
|Server Rack|  Off-Peak|        12.5|
|Server Rack|      Peak|        18.0|
+-----------+----------+------------+



Total Energy Per Device

In [16]:
device_usage = df.groupBy(
    "device_id",
    "device_name"
).agg(
    sum("energy_kwh").alias("total_energy")
)

device_usage.show()

+---------+-----------+------------+
|device_id|device_name|total_energy|
+---------+-----------+------------+
|      103| LED Lights|         5.0|
|      104|  Projector|        12.3|
|      102|Server Rack|        30.5|
|      105|    Printer|         8.0|
|      101|    AC Unit|        13.7|
+---------+-----------+------------+



Top Energy-Consuming Devices

In [17]:
top_devices = device_usage.orderBy(
    "total_energy",
    ascending=False
)

print("Top Energy Consuming Devices")

top_devices.show()

Top Energy Consuming Devices
+---------+-----------+------------+
|device_id|device_name|total_energy|
+---------+-----------+------------+
|      102|Server Rack|        30.5|
|      101|    AC Unit|        13.7|
|      104|  Projector|        12.3|
|      105|    Printer|         8.0|
|      103| LED Lights|         5.0|
+---------+-----------+------------+



In [18]:
top_devices.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("top_devices_output")

In [19]:
import os
from google.colab import files

folder = "top_devices_output"

csv_file = [f for f in os.listdir(folder) if f.endswith(".csv")][0]

files.download(os.path.join(folder, csv_file))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>